# CIS*4020 — Assignment 1 Starter Notebook
## Data Wrangling & EDA on the Open University Learning Analytics Dataset (OULAD)

This is a **scaffold** to get you started — it is **not** the solution.

- Cells marked **PROVIDED** run as-is and show you the patterns you'll need.
- Cells marked **# TODO** are yours to complete.
- Sections follow the assignment (Parts 1–5). Fill in the TODOs, justify your decisions in
  markdown, and delete these starter comments in your final submission.

**Data:** download OULAD data and point `DATA_DIR` at the CSVs
(`https://archive.ics.uci.edu/dataset/349/open+university+learning+analytics+dataset`).

## 0. Setup  *(PROVIDED)*

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)          # reproducibility (Part 5)
pd.set_option("display.max_columns", 40)
print("pandas", pd.__version__)

pandas 3.0.6


## Data loading  *(PROVIDED)*

It loads the real OULAD CSVs from `DATA_DIR` if they're present.

In [2]:
!wget https://archive.ics.uci.edu/static/public/349/open+university+learning+analytics+dataset.zip
!unzip /content/open+university+learning+analytics+dataset.zip

DATA_DIR = "/content/"   # <-- put the downloaded OULAD .csv files here for your real submission

zsh:1: command not found: wget
unzip:  cannot find or open /content/open+university+learning+analytics+dataset.zip, /content/open+university+learning+analytics+dataset.zip.zip or /content/open+university+learning+analytics+dataset.zip.ZIP.


In [3]:
FILES = {"studentInfo":"studentInfo.csv", "studentRegistration":"studentRegistration.csv",
         "studentVle":"studentVle.csv", "vle":"vle.csv", "assessments":"assessments.csv",
         "studentAssessment":"studentAssessment.csv", "courses":"courses.csv"}

def load_oulad():
    missing = [f for f in FILES.values() if not os.path.exists(os.path.join(DATA_DIR, f))]
    if missing:
        raise FileNotFoundError(
            f"Couldn't find these OULAD files in DATA_DIR={DATA_DIR!r}: {missing}. "
            "Set DATA_DIR to the folder containing the CSVs."
        )
    print("[info] loading real OULAD from", DATA_DIR)
    return {k: pd.read_csv(os.path.join(DATA_DIR, f)) for k, f in FILES.items()}

tables = load_oulad()
for name, t in tables.items():
    print(f"{name:22s} {t.shape}")

FileNotFoundError: Couldn't find these OULAD files in DATA_DIR='/content/': ['studentInfo.csv', 'studentRegistration.csv', 'studentVle.csv', 'vle.csv', 'assessments.csv', 'studentAssessment.csv', 'courses.csv']. Set DATA_DIR to the folder containing the CSVs.

## pandas quick reference

| Task | Code |
|---|---|
| read a CSV | `pd.read_csv("file.csv")` |
| shape / columns / types | `df.shape`, `df.columns`, `df.dtypes` |
| first rows | `df.head()` |
| count categories | `df["col"].value_counts()` |
| summary stats | `df.describe()` |
| filter rows | `df[df["col"] > 0]` |
| new column | `df["new"] = ...` |
| group + aggregate | `df.groupby("key").agg(total=("col","sum"))` |
| join two tables | `left.merge(right, on=["k1","k2"], how="left")` |
| count missing | `df.isna().sum()` |
| fill missing | `df["col"].fillna(value)` |
| quick plot | `df["col"].plot(kind="hist")` |


## Part 1 — Acquire & understand

In [ ]:
# PROVIDED example: how to inspect ONE table.
si = tables["studentInfo"]
print("shape (rows, columns):", si.shape)
print("\ncolumn types:")
print(si.dtypes)
print("\nfirst rows:")
display(si.head())
print("outcome breakdown:")
print(si["final_result"].value_counts())

In [ ]:
# starter for your attribute-type table — fill in a row per column:
attr_types = pd.DataFrame([
    # (table, column, pandas_dtype, attribute_type, why)
    ("studentInfo", "id_student", "int64", "nominal", "identifier; arithmetic is meaningless"),
    ("studentInfo", "imd_band",   "object", "ordinal", "deprivation bands have a natural order"),
    # ... add the rest ...
], columns=["table", "column", "pandas_dtype", "attribute_type", "why"])
attr_types

In [ ]:

# TODO (Part 1): inspect all tables the same way.
#   For each of: studentRegistration, studentVle, studentAssessment, assessments, vle, courses
#     - print its .shape and .head(), and note what one ROW represents.
#   Hint: loop over tables.items(), or access e.g. tables["studentVle"].
#
#   Then, for EACH column, state the following:
#     (1) its pandas dtype (int64, float64, object, ...), and
#     (2) its ATTRIBUTE TYPE: nominal, ordinal, interval, or ratio -- with a one-line reason.
#   Remember: the pandas dtype is NOT the attribute type. For example, id_student is stored as
#   int64 but is NOMINAL (an identifier -- arithmetic on it is meaningless), and imd_band is
#   text but is ORDINAL (the deprivation bands have a natural order).
#   You can write this as a short markdown table, or build a small DataFrame with columns
#   [table, column, pandas_dtype, attribute_type, why].


**TODO (Part 1 write-up):** In the cell below, write your **data description**:
what each table represents, how they relate (the keys), where the data comes from, and **at least three limitations** of the data.

*Your data description here...*

## Part 2 — Clean & prepare

### 2a. Aggregate the clickstream to one row per student

`studentVle` has **many rows per student** (one per resource per day). Before you can join it onto
the student table you must **reduce it to one row per student** with `groupby(...).agg(...)`.

In [ ]:
# PROVIDED worked example: total clicks and number of active days per student.
KEYS = ["code_module", "code_presentation", "id_student"]   # the join keys
vle_long = tables["studentVle"]

engagement = (vle_long
    .groupby(KEYS)                       # one group per student-module
    .agg(total_clicks=("sum_click", "sum"),     # add up all clicks
         active_days=("date", "nunique"))       # count distinct days they were active
    .reset_index())                      # turn the group keys back into columns
engagement.head()

In [ ]:
# TODO (Part 2a): add an "early engagement" feature = clicks in the first 30 days.
#   Step 1: filter the clickstream to early days:   early = vle_long[vle_long["date"] < 30]
#   Step 2: group + sum:   early.groupby(KEYS).agg(early_clicks=("sum_click","sum")).reset_index()
#   Step 3: merge it onto `engagement` (how="left").


### 2b. Join everything into one analysis table (one row per student-module)

Use a **left join** so every student is kept. Students with no clickstream rows will get `NaN`
for engagement - the correct fill there is **0** (they genuinely did nothing).

In [ ]:
# PROVIDED worked example: start the analysis table.
analysis = tables["studentInfo"].merge(engagement, on=KEYS, how="left")

# students absent from the VLE -> NaN engagement -> fill with 0 (flag them first so we don't lose info)
analysis["no_vle_activity"] = analysis["total_clicks"].isna()
for c in ["total_clicks", "active_days"]:
    analysis[c] = analysis[c].fillna(0)

# handy outcome flags for later EDA
analysis["withdrew"] = analysis["final_result"].eq("Withdrawn")
analysis["passed"]   = analysis["final_result"].isin(["Pass", "Distinction"])

print("analysis table shape:", analysis.shape)
analysis.head()

In [ ]:
# TODO (Part 2b): enrich the analysis table.
#   - merge in registration info:  tables["studentRegistration"] on KEYS (how="left")
#   - OPTIONAL: mean assessment score per student. Note studentAssessment has NO module/presentation,
#       so first join it to `assessments` on "id_assessment" to get the keys, THEN group + mean, THEN merge.


### 2c. Missing data — handle each variable *deliberately*

For every column with missing values, state the likely **mechanism** and justify impute / drop / keep-and-flag.


In [ ]:
# PROVIDED worked example: see what's missing, and handle imd_band by KEEP-AND-FLAG.
print("missing values per column:")
print(analysis.isna().sum()[analysis.isna().sum() > 0].to_string() or "  (none)")

analysis["imd_band"]    = analysis["imd_band"].replace({"10-20": "10-20%"})  # fix a known formatting quirk
analysis["imd_missing"] = analysis["imd_band"].isna()                        # flag, don't silently drop
analysis["imd_band"]    = analysis["imd_band"].fillna("Missing")             # keep as its own category
print("\nIMD missing kept as a 'Missing' category:", int(analysis["imd_missing"].sum()), "students")

In [ ]:
# TODO (Part 2c): handle date_unregistration as INFORMATIVE.
#   - it comes from tables["studentRegistration"] (merge it in 2b first).
#   - create a boolean like  analysis["unregistered"] = analysis["date_unregistration"].notna()
#   - sanity-check it against final_result == "Withdrawn" (they should mostly agree).
#   - decide what to do with date_registration if it has gaps, and JUSTIFY each choice in markdown.


**TODO (Part 2c write-up):** In the cell below, describe all your choices for dealing with missing data.

*Your data description here...*

### 2d. Outliers & inconsistencies

In [ ]:
# PROVIDED worked example: look for extreme click counts.
print(analysis["total_clicks"].describe())
q1, q3 = analysis["total_clicks"].quantile([0.25, 0.75]); iqr = q3 - q1
fence = q3 + 1.5*iqr
print(f"\nIQR upper fence = {fence:.0f}; students above it = {(analysis['total_clicks'] > fence).sum()}")

In [ ]:
# TODO (Part 2d): decide how (or whether) to treat outliers, and DOCUMENT it.
#   Are extreme click counts errors, or just very engaged students? Justify keeping/capping/removing.
#   Also check for impossible values or inconsistent categories elsewhere.


**TODO (Part 2d write-up):** In the cell below, describe all your choices for dealing with outliers.

## Part 3 — Explore (EDA)   *(need at least 5 varied visualizations)*

In [ ]:
# PROVIDED worked example 1: a bar chart of outcomes.
analysis["final_result"].value_counts().plot(kind="bar", color="#4C72B0")
plt.title("Final result"); plt.ylabel("students"); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# PROVIDED worked example 2: a histogram (engagement is very skewed -> use a log axis).
np.log1p(analysis["total_clicks"]).plot(kind="hist", bins=30, color="#55A868")
plt.title("Total clicks per student (log scale)"); plt.xlabel("log(1 + total_clicks)")
plt.tight_layout(); plt.show()

In [ ]:
# PROVIDED starter snippets you can adapt (each is a common EDA move):

# (a) a rate BY GROUP  ->  good for an equity view (Part 4 needs at least one)
pass_rate_by_imd = analysis.groupby("imd_band")["passed"].mean()
print("pass rate by IMD band:"); print(pass_rate_by_imd.round(3).to_string())

# (b) a cross-tab of two categoricals as proportions
print("\noutcome mix by disability:")
print(pd.crosstab(analysis["disability"], analysis["final_result"], normalize="index").round(2))

# (c) engagement vs outcome (a comparison)
print("\nmean total_clicks by outcome:")
print(analysis.groupby("final_result")["total_clicks"].mean().round(1).to_string())

In [ ]:
# TODO (Part 3): produce AT LEAST 5 well-chosen, varied visualizations. Ideas:
#   - pass/withdrawal rate by imd_band, disability, age_band, or prior attempts  (equity!)
#   - early engagement vs final outcome (boxplot: analysis.boxplot(column="early_clicks", by="final_result"))
#   - clicks over time: tables["studentVle"].groupby("date")["sum_click"].sum().plot()
#   - outcome mix by module or by presentation (Feb "B" vs Oct "J")
#   Aim for variety (distribution, comparison, over-time), not five bar charts.


## Part 4 — Findings (with a social lens)

In [ ]:
# PROVIDED starter: turning a group difference into a finding you can cite.
wr = analysis.groupby("imd_band")["withdrew"].mean().round(3)
print("withdrawal rate by IMD band:"); print(wr.to_string())
# ^ use numbers like these as the EVIDENCE behind each finding you write below.

**TODO (Part 4):** Write **3–5 findings**, each backed by a specific figure or statistic.
At least one must examine an **equity dimension**. For every group difference, add a sentence on
**what the data cannot tell you** (e.g., association, not causation).

1. Finding: ... *(evidence: ...)*  — caveat: ...
2. Finding: ... *(evidence: ...)*  — caveat: ...
3. Finding: ... *(evidence: ...)*  — caveat: ...

## Part 5 — Reproducibility

- The random seed is set at the top (`RANDOM_SEED`).
- Before submitting: **Kernel → Restart & Run All** to confirm the notebook runs top-to-bottom.
- State clearly where the data files go (how your `DATA_DIR`is set);

## Submission checklist
- [ ] `DATA_DIR` points at the real OULAD files
- [ ] Every cleaning decision is justified in a markdown cell
- [ ] At least 5 varied visualizations
- [ ] 3–5 findings including one equity finding, each with a "what the data can't tell us" caveat
- [ ] Runs top-to-bottom on a clean kernel
- [ ] AI use disclosed (see assignment notes)